# PAX → Delta (Phase B — direct ingest, locked path)

Pulls Purview Audit + Entra data directly from Microsoft Graph and writes
**Delta tables** under `Tables/<TargetSchema>/` in the default lakehouse.
**No persistent CSV output** — the CSV bundle from Phase A0 / Phase A is
not produced. (A transient scratch directory is used internally so the
existing rollup / Entra processors can be reused; it is deleted at end
of run unless `KeepScratch=True`.)

## Prerequisites
- Default lakehouse attached (schema-enabled), with the target schema
  (default `dbo`) already created.
- **`pax_fabric/` package is pulled from GitHub at run time** — no manual
  upload to `Files/code/` is required. The clone cell below does a fresh
  shallow `git clone` of `GitHubRepoUrl` at `GitHubRef` (the branch to
  pull from) into a temp dir on every run, so each execution is free of
  stale `.pyc` from prior runs. The temp dir is removed by the cleanup
  cell at the end.
- A **variable library** is attached to the workspace and supplies the
  parameters consumed by the parameters cell below (auth identifiers,
  Key Vault location, date range, run-control flags, and the GitHub
  repo URL + branch). The client secret is fetched from Key Vault via
  `notebookutils.credentials.getSecret(KeyVaultUri, KeyVaultSecret)`.
- The repo branch in `GitHubRef` must be reachable from the Fabric
  runtime (public repo, or private repo with credentials baked into
  the URL / the runtime's git config).
- `git` is on `PATH` in the Fabric Python runtime (it is, by default).
- `deltalake>=0.17` available (auto-installed by the clone cell if
  missing).

## Output
- Delta tables written under `Tables/<TargetSchema>/` in the default
  lakehouse. Which tables are produced depends on the run flags:
  - **Audit raw** — always written when audit ingest runs.
  - **CopilotInteraction rollup family** — `..._Rollup`,
    plus `..._UserStats`, `..._SessionCohort`, and `..._SessionStats`
    when `IncludeM365Usage=True`.
  - **Entra users** — `Entra_Users` when `IncludeUserInfo=True`.
- **Append semantics:** every run appends new rows to the same target
  table per logical dataset. The function returns one entry per
  written table in `result["delta_tables"]` with `rows_written` and
  `is_init` (true on the first write that creates the table).
- **Schema evolution (mod16):**
  - **Additive drift** — new columns in the incoming batch are
    absorbed automatically via `schema_mode='merge'`; existing rows
    get NULL for the new column.
  - **Destructive drift** — when columns the Delta table already has
    are missing from the incoming batch, the auto-drop path drops
    those columns from the Delta table (regardless of how many) and
    then appends. Delta time-travel preserves the pre-drop version
    for ~7 days, so an accidental drop is recoverable via
    `DeltaTable(uri).restore(version=N)`.
- **Optional retention** (`RetentionDays`) prunes rows older than the
  cutoff from each Delta table that has a `CreationDate` column. When
  retention deletes any rows from a `_Rollup` table and
  `IncludeM365Usage=True`, `UserStats` and `SessionCohort` are
  recomputed from the trimmed Rollup so percentiles, tiers, and
  active-day counts reflect only the retained window.

## Gate
The verify cell reads each written Delta table back and asserts the
post-write row count is **at least** `rows_written` (i.e. the table
exists, is readable, and contains the rows we just appended).

In [ ]:
# ---- Parameters (the only cell you should edit run-to-run) ----

import notebookutils

vl = notebookutils.variableLibrary.getLibrary("PAX_VarLib")

# Auth
TenantId     = vl.TenantId
ClientId     = vl.ClientId

# Secret from Key Vault
KeyVaultUri    = vl.KeyVaultUri
KeyVaultSecret = vl.KeyVaultSecret

ClientSecret = notebookutils.credentials.getSecret(KeyVaultUri, KeyVaultSecret)

if not ClientSecret:
    raise RuntimeError(
        f"Failed to fetch secret '{KeyVaultSecret}' from {KeyVaultUri}. "
        "Verify Key Vault access and secret name."
    )
print(f"Secret fetched from Key Vault: {KeyVaultSecret} (length={len(ClientSecret)})")


# Date range
StartDate = vl.StartDate
EndDate   = vl.EndDate

# Run controls
Rollup           = vl.Rollup
TargetSchema     = vl.TargetSchema
KeepScratch      = vl.KeepScratch
IncludeM365Usage = vl.IncludeM365Usage
IncludeUserInfo  = vl.IncludeUserInfo

RetentionDays = None

# Path to the pax_fabric package in the lakehouse
# CodePath = vl.CodePath

# GitHub source for pax_fabric package
GitHubRepoUrl = vl.GitHubRepoUrl   # e.g. https://github.com/<owner>/<repo>.git
GitHubRef     = vl.GitHubRef       # branch, tag, or commit SHA (pin a tag/SHA for prod)

# print(CodePath)


In [ ]:
# ---- Clone pax_fabric from GitHub (public repo) ----
import os, sys, subprocess, tempfile, importlib

# Fresh clone each run — no stale .pyc, deterministic per GitHubRef
CodePath = tempfile.mkdtemp(prefix="pax_fabric_repo_")
subprocess.check_call(
    ["git", "clone", "--depth", "1", "--branch", GitHubRef, GitHubRepoUrl, CodePath],
    stdout=subprocess.DEVNULL,
)
print(f"Cloned {GitHubRepoUrl}@{GitHubRef} -> {CodePath}")

# pax_fabric/ is at the repo root, so the parent dir on sys.path is CodePath itself
PackageParent = CodePath

# Evict any prior pax_fabric import so a re-run picks up the new clone
for _m in [m for m in list(sys.modules) if m == "pax_fabric" or m.startswith("pax_fabric.")]:
    del sys.modules[_m]

# ---- Dependencies: ensure deltalake is available ----
try:
    import deltalake  # noqa: F401
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "deltalake>=0.17"]
    )
    importlib.invalidate_caches()
    import deltalake  # noqa: F401

import pyarrow  # noqa: F401
import pandas   # noqa: F401
print("deltalake version:", deltalake.__version__)

In [ ]:
# ---- Invoke pax_fabric.run() ----
import sys

# if PackageParent and PackageParent not in sys.path:
#     sys.path.insert(0, PackageParent)

if CodePath and CodePath not in sys.path:
    sys.path.insert(0, CodePath)

from pax_fabric import run as pax_run, files_io  # noqa: E402

params = {
    "Auth":         "AppRegistration",
    "TenantId":     TenantId,
    "ClientId":     ClientId,
    "ClientSecret": ClientSecret,
    "StartDate":    StartDate,
    "EndDate":      EndDate,
    "Rollup":       Rollup,
    "OutputMode":   "delta",
    "TargetSchema": TargetSchema,
    "KeepScratch":  KeepScratch,
    "IncludeM365Usage": IncludeM365Usage,
    "IncludeUserInfo": IncludeUserInfo,
}

result = pax_run(params)

print(f"success       : {result['success']}")
print(f"run_id        : {result['run_id']}")
print(f"target_schema : {result['target_schema']}")
print(f"records       : {result['records_fetched']}")
print(f"output_rows   : {result['output_rows']}")
print(f"elapsed (s)   : {result['elapsed_seconds']}")
print(f"log_file      : {result['log_file']}")
print()
print("Delta tables written:")
for t in result.get("delta_tables", []):
    print(f"  - {t['table']:<50} rows={t['rows_written']:<8} path={t['path']}")

if not result["success"]:
    raise RuntimeError(f"pax_run failed: {result.get('error')}")

In [ ]:
# ---- Phase B gate: read each Delta table back and confirm rows are present ----
from deltalake import DeltaTable

storage_options = files_io.onelake_storage_options()

failures = []
print(f"{'TABLE':<55} {'WRITTEN':>10} {'DELTA_TOTAL':>12}  STATUS")
print("-" * 92)
for t in result.get("delta_tables", []):
    written = t["rows_written"]
    try:
        dt = DeltaTable(t["path"], storage_options=storage_options)
        total_rows = dt.to_pyarrow_dataset().count_rows()
    except Exception as exc:
        print(f"{t['table']:<55} {written:>10}   ERROR reading back: {exc}")
        failures.append(t["table"])
        continue
    # Post-append total must be >= what we just wrote. On first ingest
    # (t['is_init'] == True) this is equality; on subsequent runs it is
    # strictly greater because prior rows accumulate.
    flag = "OK" if total_rows >= written else "MISMATCH"
    init_tag = " [append-init]" if t.get("is_init") else ""
    print(f"{t['table']:<55} {written:>10} {total_rows:>12}  {flag}{init_tag}")
    if total_rows < written:
        failures.append(t["table"])

if failures:
    raise AssertionError(
        f"Phase B gate FAILED for {len(failures)} table(s): {failures}"
    )

print()
print("Phase B gate: all tables readable with row count >= rows written this run.")



In [ ]:
# ---- Data Retention: prune rows older than RetentionDays ----
if RetentionDays is not None and RetentionDays > 0:
    from pax_fabric.mod17_pax_retention import enforce_retention
 
    storage_options = files_io.onelake_storage_options()
 
    print(f"Data retention: deleting rows with CreationDate older than {RetentionDays} days")
    print()
    print(f"{'TABLE':<55} {'BEFORE':>10} {'AFTER':>10} {'DELETED':>10}  STATUS")
    print("-" * 100)
 
    retention_failures = []
    _rollup_rows_deleted = 0
    for t in result.get("delta_tables", []):
        ret = enforce_retention(
            target_uri=t["path"],
            retention_days=RetentionDays,
            storage_options=storage_options,
        )
 
        if ret["skipped"]:
            print(f"{t['table']:<55} {'—':>10} {'—':>10} {'—':>10}  SKIPPED (no CreationDate)")
            continue
 
        if not ret["success"]:
            print(f"{t['table']:<55} {ret['rows_before']:>10} {'—':>10} {'—':>10}  ERROR: {ret['error']}")
            retention_failures.append(t["table"])
            continue
 
        print(
            f"{t['table']:<55} {ret['rows_before']:>10} {ret['rows_after']:>10} "
            f"{ret['rows_deleted']:>10}  OK (cutoff: {ret['cutoff_date']})"
        )
 
        # Track if Rollup Delta had rows deleted (triggers recompute below)
        if "_Rollup" in t["table"] and "UserStats" not in t["table"] and "SessionCohort" not in t["table"] and "SessionStats" not in t["table"]:
            _rollup_rows_deleted += ret.get("rows_deleted", 0)
 
    if retention_failures:
        print()
        print(f"⚠ Retention failed for {len(retention_failures)} table(s): {retention_failures}")
    else:
        print()
        print(f"Data retention complete (cutoff: {RetentionDays} days).")
 
    # ---- Post-retention recompute: refresh UserStats & SessionCohort ----
    # After retention trims old rows from Rollup/SessionStats Delta, the
    # UserStats and SessionCohort tables are stale (computed from pre-trim
    # data). Recompute them from the trimmed Rollup so percentiles, tiers,
    # and active-day counts reflect only the retained window.
    # Only runs when: Rollup rows were actually deleted AND M365 usage mode.
    if _rollup_rows_deleted > 0 and IncludeM365Usage:
        print()
        print(f"Post-retention recompute: Rollup had {_rollup_rows_deleted:,} rows deleted — "
              "refreshing UserStats & SessionCohort from trimmed data...")
        from pax_fabric.pipeline import _recompute_userstats_from_delta
        recomputed = _recompute_userstats_from_delta(
            delta_results=result.get("delta_tables", []),
            schema=TargetSchema,
            log_fn=lambda msg, lvl="INFO": print(f"  {msg}"),
        )
        if recomputed:
            print(f"Post-retention recompute: {len(recomputed)} table(s) refreshed.")
            for r in recomputed:
                print(f"  {r['table']:<55} {r['rows_written']:>10} rows  [recomputed]")
        else:
            print("Post-retention recompute: no tables refreshed (check logs above for warnings).")
else:
    print("Data retention: disabled (RetentionDays is None or 0).")

In [ ]:
# ---- Cleanup: remove the GitHub clone temp dir ----
import os, shutil

if 'CodePath' in dir() and CodePath and os.path.isdir(CodePath):
    try:
        shutil.rmtree(CodePath)
        print(f"Cleanup: removed temp clone dir {CodePath}")
    except Exception as exc:
        print(f"Cleanup: failed to remove {CodePath} -> {exc}")
else:
    print("Cleanup: no temp clone dir to remove (already gone or not set).")